# ResNet-20 / CIFAR-10: a slow walk through the sticky PDMP samples

One run so far: **Sticky Zig-Zag**, ResNet-20 on a CIFAR-10 subset, sampled
for **1,000,000 skeleton events** (`fast_cheap_cifar_resnet.py`, staged).

- `results/paper/cifar/grid_sticky_zigzag.pt`  -- Sticky Zig-Zag

This is the ResNet-20 analogue of `lenet_mnist.ipynb`, and it follows the
same order. Two things differ from the LeNet5 notebook and shape every
section:

- **BatchNorm.** ResNet-20 has 21 BatchNorm2d layers. Their `weight`
  (gamma) and `bias` (beta) are sampled; their `running_mean` /
  `running_var` are frozen buffers (populated by pretraining, module in
  `.eval()` for the whole pipeline). Gamma is **never** eligible to freeze
  (`build_can_freeze_mask_resnet`), so the sparsity story is about
  conv/linear *weights* only.
- **No dead border.** CIFAR-10 images fill the 32x32 frame edge to edge, and
  the stem conv is `3x3, padding=1` -- there is no always-black ring for the
  input layer to ignore. Section 3 therefore asks a different question of
  the stem conv: per-RGB-channel structure and where posterior activation
  uncertainty lands on a real image.

Order:

1. **What is in the file?** Every key, shape, dtype, value range, what is
   present and what is missing.
2. **Layer map**: unflatten the 272,474 coordinates back to
   `(layer, position)`, split conv / BatchNorm-gamma / BatchNorm-beta / fc.
3. **The stem conv** (`stem_conv.weight`, `[16, 3, 3, 3]`): per-channel mean
   kernels, exact-zero rate, and the posterior activation-uncertainty map on
   a real CIFAR image.
4. **Sparsity across the resampled path**, and never-left-zero vs moved by
   layer.
5. **Displacement from the MAP reference** (`x_ref`), in prior-std units,
   per layer.
6. **Predictive accuracy and uncertainty**: posterior-averaged predictions
   vs the single MAP point, reliability, entropy on correct vs wrong.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
%matplotlib inline

if Path.cwd().name == "notebooks":
    os.chdir("..")

plt.rcParams.update({
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.alpha":         0.3,
    "font.size":          11,
})

RUN_DIR = Path("results/paper/cifar")
RUN_SPECS = [
    ("zigzag", "grid_sticky_zigzag.pt"),
    ("boomerang", "grid_sticky_boomerang.pt"),   # not present yet -- skipped if missing
]
RUN_DISPLAY_NAME = {"zigzag": "Sticky Zig-Zag", "boomerang": "Sticky Boomerang"}
print("cwd:", Path.cwd())

## 1. What is in the file?

### 1a. File-level: size, top-level type, every key

`torch.load(..., weights_only=False)` because these checkpoints hold Python
scalars and lists next to the tensors. For each key we print: tensor shape /
dtype / min / max / a near-zero fraction for the big ones, or the raw value
for scalars, or length + head for lists.

In [ ]:
def describe_ckpt(ck):
    rows = []
    for k, v in ck.items():
        if torch.is_tensor(v):
            near0 = float((v.abs() < 1e-8).float().mean()) if v.dtype.is_floating_point else np.nan
            rows.append({
                "key": k, "kind": "tensor", "shape": str(tuple(v.shape)),
                "dtype": str(v.dtype),
                "min": float(v.min()) if v.numel() else np.nan,
                "max": float(v.max()) if v.numel() else np.nan,
                "frac |.|<1e-8": near0,
            })
        elif isinstance(v, (list, tuple)):
            rows.append({"key": k, "kind": type(v).__name__, "shape": f"len={len(v)}",
                         "dtype": "", "min": np.nan, "max": np.nan, "frac |.|<1e-8": np.nan,
                         "value/head": repr(v[:3])})
        elif isinstance(v, dict):
            rows.append({"key": k, "kind": "dict", "shape": f"{len(v)} keys",
                         "dtype": "", "min": np.nan, "max": np.nan, "frac |.|<1e-8": np.nan,
                         "value/head": repr(list(v.keys())[:8])})
        else:
            rows.append({"key": k, "kind": type(v).__name__, "shape": "", "dtype": "",
                         "min": np.nan, "max": np.nan, "frac |.|<1e-8": np.nan,
                         "value/head": repr(v)})
    return pd.DataFrame(rows).set_index("key")


runs = {}
for label, fname in RUN_SPECS:
    path = RUN_DIR / fname
    if not path.exists():
        print(f"[{label}] MISSING -- {path}  (skipping)")
        continue
    size_gb = path.stat().st_size / 1e9
    ck = torch.load(path, map_location="cpu", weights_only=False)
    runs[label] = {"ckpt": ck, "D": int(ck["x_ref"].shape[0]), "path": path, "size_gb": size_gb}
    print("=" * 78)
    print(f"[{label}]  {fname}   {size_gb:.2f} GB on disk   top-level type: {type(ck).__name__}")
    print("=" * 78)
    display(describe_ckpt(ck))

assert runs, f"No run files under {RUN_DIR}"

### 1b. What do we have, and what is missing?

Plain-language summary of the important fields, plus an explicit list of what
is *not* in this checkpoint -- so later sections do not silently assume it.

In [ ]:
for label, r in runs.items():
    ck = r["ckpt"]
    n_draws, D = ck["samples"].shape
    print(f"[{RUN_DISPLAY_NAME[label]}]")
    print(f"  sampler              : {ck['sampler']}")
    print(f"  architecture         : ResNet-20  (activation={ck['activation']}, pool={ck['pool']})")
    print(f"  D (params)           : {D}")
    print(f"  skeleton events run  : {ck['n_events']:,}")
    print(f"  resampled draws saved: {n_draws:,}   -> samples tensor is [{n_draws}, {D}]")
    print(f"  wall time            : {ck['elapsed_sec']:.1f} s  ({ck['elapsed_sec']/3600:.2f} h)")
    print(f"  gradient evals       : {ck['gradient_evals']:,}")
    print(f"  bound_violations     : {ck['bound_violations']}")
    print(f"  prune_frac (t=0)     : {ck['prune_frac']:.4f}   (fraction of freezable coords cold-start frozen)")
    print(f"  sparsity_frac (saved): {ck['sparsity_frac']:.4f}   (near-zero fraction over resampled draws)")
    print(f"  test_accuracy (saved): {ck['test_accuracy']:.4f}")
    cs = ck['cold_start_mask']
    print(f"  cold_start_mask      : bool[{cs.shape[0]}], "
          f"{int(cs.sum())} True ({100*cs.float().mean():.2f}%)")
    print(f"  grid_t_max_log       : list, len {len(ck['grid_t_max_log'])}  "
          f"(one entry per _grid_bound call; head={ck['grid_t_max_log'][:2]} tail={ck['grid_t_max_log'][-2:]})")
    diag = ck.get("diagnostics")
    print(f"  diagnostics          : {type(diag).__name__}"
          + ("" if diag is None else f"  ({len(diag)} rows)"))
    print()

print("NOT in this checkpoint (so the corresponding analysis is unavailable):")
print("  - per-iteration diagnostics list (diagnostics is None -- fast_cheap_*")
print("    saves diagnostics=None by construction). No per-coordinate bounce")
print("    attribution, no freeze/thaw/bounce event mix, no per-iteration max_ratio.")
print("  - the full skeleton path: only the resampled draws are stored, not all 1e6 events.")
print("  - a Sticky Boomerang run (only Zig-Zag has been sampled so far).")

### 1c. The `samples` tensor up close

`samples[i]` is one full parameter vector (one posterior draw); `samples[:, j]`
is the marginal path of coordinate `j` across the resampled draws.

In [ ]:
for label, r in runs.items():
    s = r["ckpt"]["samples"]
    print(f"[{RUN_DISPLAY_NAME[label]}] samples: shape={tuple(s.shape)} dtype={s.dtype} "
          f"contiguous={s.is_contiguous()} mem={s.element_size()*s.nelement()/1e6:.1f} MB")
    exact_zero = (s == 0)
    print(f"    exact zeros: {int(exact_zero.sum()):,} / {s.nelement():,} "
          f"({100*exact_zero.float().mean():.2f}%)   "
          f"rows with >=1 exact zero: {int(exact_zero.any(1).sum())}/{s.shape[0]}")
    per_draw_near0 = (s.abs() < 1e-8).float().mean(1)
    print(f"    per-draw near-zero (|.|<1e-8) fraction: "
          f"min={per_draw_near0.min().item():.4f}  max={per_draw_near0.max().item():.4f}")
    col_std = s.std(0)
    print(f"    per-coordinate std across draws: "
          f"median={col_std.median().item():.4g}  p95={col_std.quantile(0.95).item():.4g}  "
          f"max={col_std.max().item():.4g}")
    print(f"    fully-static coordinates (std==0 across all draws): "
          f"{int((col_std == 0).sum()):,} / {s.shape[1]:,}")
    print()

## 2. Layer map: unflatten coordinates back to `(layer, position)`

Every coordinate `0..272473` belongs to one ResNet-20 parameter tensor, in
`module.named_parameters()` order. We build the map straight off a fresh
`ResNet20` so it stays in lock-step with `neural_networks.py`, and tag each
coordinate as `conv` / `bn_weight` (gamma) / `bn_bias` (beta) / `fc` -- the
four kinds the priors / freeze-mask builders treat differently.

In [ ]:
from sazz.gpu_friendly.models.neural_networks import ResNet20
from sazz.gpu_friendly.models.priors import _is_batchnorm_weight

_ref_module = ResNet20(activation="relu")
D_EXPECTED = sum(p.numel() for p in _ref_module.parameters())
assert D_EXPECTED == 272474, D_EXPECTED


def _kind(module, name, p):
    if _is_batchnorm_weight(module, name):
        return "bn_weight"
    if name.endswith(".bias") and p.dim() == 1:
        # a BN beta, or the single fc bias
        sub = name[: -len(".bias")]
        submod = module.get_submodule(sub) if sub else module
        return "bn_bias" if isinstance(submod, torch.nn.modules.batchnorm._BatchNorm) else "fc"
    if name.startswith("fc."):
        return "fc"
    return "conv"


def build_layer_index(module):
    rows, idx = [], 0
    for name, p in module.named_parameters():
        shape = tuple(p.shape)
        n = p.numel()
        kind = _kind(module, name, p)
        for local_i in range(n):
            pos = tuple(int(x) for x in np.unravel_index(local_i, shape)) if len(shape) > 1 else (local_i,)
            rows.append({"coord": idx, "layer": name, "kind": kind, "shape": shape, "pos": pos})
            idx += 1
    return pd.DataFrame(rows).set_index("coord")


layer_idx = build_layer_index(_ref_module)
assert len(layer_idx) == D_EXPECTED

by_layer = layer_idx.groupby("layer", sort=False).agg(
    kind=("kind", "first"), n=("layer", "size"),
    first_coord=("kind", lambda s: s.index.min()), last_coord=("kind", lambda s: s.index.max()),
)
display(by_layer)
print("coordinate count by kind:")
print(layer_idx["kind"].value_counts())

### 2b. Sparsity is a conv/fc story only

`build_can_freeze_mask_resnet` marks every 1-D parameter (BatchNorm gamma
AND beta, plus the fc bias) as non-freezable -- only >=2-D conv/linear
*weights* can ever be stuck to exactly zero. Confirm the checkpoint's
`cold_start_mask` respects that, and see how the cold-start prune splits
across layers.

In [ ]:
from sazz.gpu_friendly.models.priors import build_can_freeze_mask_resnet

can_freeze = build_can_freeze_mask_resnet(_ref_module)
print(f"freezable coords: {int(can_freeze.sum()):,} / {len(can_freeze):,} "
      f"({100*can_freeze.float().mean():.2f}%)  -- the rest are BN gamma/beta + fc bias")

for label, r in runs.items():
    cs = r["ckpt"]["cold_start_mask"]
    frozen_non_freezable = int((cs & ~can_freeze).sum())
    print(f"\n[{RUN_DISPLAY_NAME[label]}] cold_start_mask: {int(cs.sum()):,} frozen")
    print(f"    frozen coords that are NOT freezable: {frozen_non_freezable}  (must be 0)")
    ldf = layer_idx.copy()
    ldf["cold"] = cs.numpy()
    ldf["freezable"] = can_freeze.numpy()
    g = ldf.groupby(["kind"], sort=False).agg(
        n=("cold", "size"), freezable=("freezable", "sum"), cold_frozen=("cold", "sum"))
    g["frac_frozen_of_freezable"] = g["cold_frozen"] / g["freezable"].replace(0, np.nan)
    display(g.style.format({"frac_frozen_of_freezable": "{:.3f}"}, na_rep="-"))

## 3. The stem conv: `stem_conv.weight`, `[16, 3, 3, 3]`

The stem conv is the only layer that touches raw RGB pixels
(`Conv2d(3, 16, 3, padding=1, bias=False)`, then BatchNorm, then ReLU).
Unlike MNIST there is no always-black border to test against, so we ask:

- **3a.** For each of the 16 output filters, the posterior-mean `3x3` kernel
  per input channel (R, G, B), the exact-zero rate per tap, and the per-tap
  std across draws. Does the sampler zero whole (filter, channel) slabs?
- **3b.** Push `N` posterior draws of `stem_conv` (+ its BatchNorm gamma/beta
  and frozen running stats) through the stem on one real CIFAR image, and
  look at where the across-draw activation std concentrates.

In [ ]:
from torchvision import datasets, transforms

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)
CIFAR10_CLASSES = ["plane", "car", "bird", "cat", "deer",
                   "dog", "frog", "horse", "ship", "truck"]

_norm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
_cifar_train = datasets.CIFAR10("datasets", train=True, download=True, transform=_norm)

_mean_t = torch.tensor(CIFAR10_MEAN).view(3, 1, 1)
_std_t = torch.tensor(CIFAR10_STD).view(3, 1, 1)

def _denorm(x):
    """normalised [3,32,32] tensor -> [32,32,3] in [0,1] for imshow."""
    return (x * _std_t + _mean_t).clamp(0, 1).permute(1, 2, 0).numpy()

_rng = np.random.default_rng(0)
_sample_idx = _rng.choice(len(_cifar_train), size=8, replace=False)
_demo_x, _demo_y = _cifar_train[int(_sample_idx[3])]   # one image reused below
print("demo image class:", CIFAR10_CLASSES[_demo_y])

fig, ax = plt.subplots(1, 8, figsize=(14, 2))
for a, i in zip(ax, _sample_idx):
    x, y = _cifar_train[int(i)]
    a.imshow(_denorm(x)); a.set_title(CIFAR10_CLASSES[y], fontsize=9)
    a.set_xticks([]); a.set_yticks([])
fig.suptitle("CIFAR-10 sample (de-normalised)"); fig.tight_layout(); plt.show()

### 3a. Posterior-mean stem kernels and per-tap exact-zero rate

`stem_conv.weight` is `[16, 3, 3, 3]` -> 432 coordinates. Reshape all draws
to `[n_draws, 16, 3, 3, 3]`; show, per filter, the mean `3x3` kernel for each
RGB channel (top rows) and the exact-zero rate per tap (bottom rows).

In [ ]:
for label, r in runs.items():
    w_coords = torch.as_tensor(layer_idx.index[layer_idx["layer"] == "stem_conv.weight"].to_numpy())
    s = r["ckpt"]["samples"][:, w_coords].view(-1, 16, 3, 3, 3).float()  # [n,16,3,3,3]

    mean_k = s.mean(0)                    # [16,3,3,3]
    zero_rate = (s == 0).float().mean(0)  # [16,3,3,3]
    std_k = s.std(0)                      # [16,3,3,3]
    slab_zero = (s == 0).all(0)           # [16,3,3,3] taps zero in EVERY draw

    # per (filter, channel) slab: fraction of the 9 taps that are zero in every draw
    slab_frac = slab_zero.float().mean(dim=(2, 3))  # [16,3]

    fig, axes = plt.subplots(4, 16, figsize=(20, 5.2))
    vlim = mean_k.abs().max()
    for f in range(16):
        # top 3 rows: mean kernel per RGB channel
        for c in range(3):
            axes[c, f].imshow(mean_k[f, c], cmap="RdBu_r", vmin=-vlim, vmax=vlim)
            if f == 0:
                axes[c, f].set_ylabel("RGB"[c], fontsize=10)
        axes[0, f].set_title(f"filt {f}", fontsize=8)
        # bottom row: mean exact-zero rate over the 3x3x3 = 27 taps
        axes[3, f].imshow(zero_rate[f].mean(0), cmap="Greys", vmin=0, vmax=1)
        axes[3, f].set_title(f"{zero_rate[f].mean():.2f}", fontsize=8)
    axes[3, 0].set_ylabel("zero rate", fontsize=9)
    for a in axes.flat:
        a.set_xticks([]); a.set_yticks([])
    fig.suptitle(f"{RUN_DISPLAY_NAME[label]}: stem_conv.weight -- mean kernel per RGB channel (rows R/G/B) "
                 f"+ mean exact-zero rate (bottom)", fontsize=12)
    fig.tight_layout(); plt.show()

    print(f"[{RUN_DISPLAY_NAME[label]}] stem_conv.weight: "
          f"overall exact-zero rate {(s == 0).float().mean():.3f}, "
          f"taps zero in EVERY draw: {int(slab_zero.sum())}/432, "
          f"taps never zero: {int((s != 0).all(0).sum())}/432")
    fig, ax = plt.subplots(figsize=(7, 3))
    im = ax.imshow(slab_frac.numpy(), cmap="magma", vmin=0, vmax=1, aspect="auto")
    ax.set(xlabel="input channel (R,G,B)", ylabel="filter", xticks=[0, 1, 2],
           xticklabels=["R", "G", "B"], title="fraction of the 9 taps zero in EVERY draw, per (filter, channel)")
    fig.colorbar(im, ax=ax, fraction=0.046); fig.tight_layout(); plt.show()

### 3b. Posterior uncertainty in the stem activations on a real image

Push `N_DRAWS_SHOWN` posterior draws of the **stem** (`stem_conv.weight` +
`stem_bn.weight` + `stem_bn.bias`, with the frozen `stem_bn` running
stats from the reference module) through `stem -> BN -> ReLU` on one real
CIFAR image. Look at the per-position mean and across-draw std of the 16
post-ReLU activation maps.

In [ ]:
import torch.nn.functional as F

N_DRAWS_SHOWN = 300

# frozen BatchNorm running stats for the stem, taken from a reference checkpoint's
# module_state_dict (all three resnet map checkpoints carry identical stem_bn stats).
_MREF = Path("results/maps/resnet/resnet20_reference_N50000_steps2000_pruned_refit.pt")
_msd = torch.load(_MREF, map_location="cpu", weights_only=False)["module_state_dict"]
stem_run_mean = _msd["stem_bn.running_mean"].float()
stem_run_var = _msd["stem_bn.running_var"].float()
print(f"stem_bn running_mean |.|max={stem_run_mean.abs().max():.3f}  "
      f"running_var in [{stem_run_var.min():.3f}, {stem_run_var.max():.3f}]")

_img = _demo_x.unsqueeze(0).float()  # [1,3,32,32] normalised

for label, r in runs.items():
    wc = torch.as_tensor(layer_idx.index[layer_idx["layer"] == "stem_conv.weight"].to_numpy())
    gc = torch.as_tensor(layer_idx.index[layer_idx["layer"] == "stem_bn.weight"].to_numpy())
    bc = torch.as_tensor(layer_idx.index[layer_idx["layer"] == "stem_bn.bias"].to_numpy())
    s = r["ckpt"]["samples"]
    n_draws = min(N_DRAWS_SHOWN, s.shape[0])
    draw_idx = torch.randperm(s.shape[0])[:n_draws]

    acts = []
    with torch.no_grad():
        for i in draw_idx:
            w = s[i, wc].view(16, 3, 3, 3).float()
            gamma = s[i, gc].float()
            beta = s[i, bc].float()
            z = F.conv2d(_img, w, bias=None, padding=1)               # [1,16,32,32]
            z = F.batch_norm(z, stem_run_mean, stem_run_var, gamma, beta,
                             training=False, eps=1e-5)
            acts.append(F.relu(z).squeeze(0))                          # [16,32,32]
    acts = torch.stack(acts)  # [n_draws,16,32,32]

    mean_act = acts.mean(0)
    std_act = acts.std(0)

    fig, axes = plt.subplots(2, 9, figsize=(16, 4))
    axes[0, 0].imshow(_denorm(_demo_x)); axes[0, 0].set_title(f"input: {CIFAR10_CLASSES[_demo_y]}", fontsize=9)
    axes[1, 0].axis("off")
    axes[1, 0].text(0.5, 0.5, RUN_DISPLAY_NAME[label], ha="center", va="center",
                    fontsize=11, fontweight="semibold", transform=axes[1, 0].transAxes)
    for c in range(8):
        axes[0, c + 1].imshow(mean_act[c], cmap="viridis"); axes[0, c + 1].set_title(f"ch{c} mean", fontsize=9)
        axes[1, c + 1].imshow(std_act[c], cmap="viridis"); axes[1, c + 1].set_title(f"ch{c} std", fontsize=9)
    for a in axes.flat:
        a.set_xticks([]); a.set_yticks([])
    fig.suptitle(f"{RUN_DISPLAY_NAME[label]}: stem post-ReLU activation -- mean (top) and across-draw std (bottom), first 8 of 16 channels",
                 fontsize=12)
    fig.tight_layout(); plt.show()

    # summary: does std track mean activation (signal), or is it flat?
    m = mean_act.flatten().numpy(); sd = std_act.flatten().numpy()
    from scipy.stats import spearmanr
    rho, _ = spearmanr(m, sd)
    print(f"[{RUN_DISPLAY_NAME[label]}] spearman(mean act, across-draw std) = {rho:.3f}   "
          f"std: median={np.median(sd):.4g}  p95={np.quantile(sd, 0.95):.4g}")

## 4. Sparsity across the resampled path

`prune_frac` is the cold-start frozen fraction of freezable coords (t=0).
`sparsity_frac` is the near-zero fraction over the resampled draws. The
per-draw curve shows whether the sampler thawed (sparsity falls) or froze
further (rises) over the run.

Note the resampled draws are concatenated **per stage** (equal draws per
stage, `fast_cheap_cifar_resnet.py`), so the x-axis is roughly ordered in
sampler time, stage by stage.

In [ ]:
rows = []
for label, r in runs.items():
    ck = r["ckpt"]
    per_draw = (ck["samples"].abs() < 1e-8).float().mean(1)
    rows.append({
        "run": RUN_DISPLAY_NAME[label],
        "prune_frac (t=0)": ck["prune_frac"],
        "sparsity_frac (saved)": ck["sparsity_frac"],
        "per-draw sparsity min": float(per_draw.min()),
        "per-draw sparsity mean": float(per_draw.mean()),
        "per-draw sparsity max": float(per_draw.max()),
        "bound_violations": ck["bound_violations"],
        "wall h": ck["elapsed_sec"] / 3600,
    })
display(pd.DataFrame(rows).set_index("run").style.format({
    "prune_frac (t=0)": "{:.4f}", "sparsity_frac (saved)": "{:.4f}",
    "per-draw sparsity min": "{:.4f}", "per-draw sparsity mean": "{:.4f}",
    "per-draw sparsity max": "{:.4f}", "wall h": "{:.2f}",
}))

fig, ax = plt.subplots(figsize=(9, 3.6))
for label, r in runs.items():
    per_draw = (r["ckpt"]["samples"].abs() < 1e-8).float().mean(1)
    ax.plot(per_draw.numpy(), lw=0.9, label=RUN_DISPLAY_NAME[label])
    ax.axhline(r["ckpt"]["prune_frac"] * float(can_freeze.float().mean()), ls="--", lw=0.8,
               alpha=0.6, color="grey",
               label=f"{RUN_DISPLAY_NAME[label]} cold-start sparsity (of full D)")
ax.set(xlabel="resampled draw index (stage-ordered)", ylabel="fraction of D with |param| < 1e-8",
       title="Sparsity along the resampled path", ylim=(0, 1))
ax.legend(fontsize=8)
fig.tight_layout(); plt.show()

### 4b. Never-left-zero vs moved, by layer

No per-iteration diagnostics were saved, so no bounce/thaw event mix. What we
*can* read from `samples` + `cold_start_mask`:

- **never left zero** : cold-start frozen AND every draw is (near-)zero.
- **moved**           : left near-zero in at least one draw.

A layer almost entirely "never left zero" is a stuck / genuinely-flat region.
BN gamma/beta and the fc bias are never cold-start frozen, so any of those in
"never left zero" would have to have been frozen dynamically (should be none).

In [ ]:
mask_rows = []
by_layer_frames = {}
for label, r in runs.items():
    ck = r["ckpt"]
    s = ck["samples"]
    cold = ck["cold_start_mask"]
    ever_nonzero = (s.abs() > 1e-8).any(0)
    never_left_zero = cold & ~ever_nonzero
    moved = ever_nonzero

    ldf = layer_idx.copy()
    ldf["never_left_zero"] = never_left_zero.numpy()
    ldf["moved"] = moved.numpy()
    ldf["cold"] = cold.numpy()
    g = ldf.groupby("layer", sort=False).agg(
        kind=("kind", "first"), n=("layer", "size"),
        cold_frozen=("cold", "sum"), never_left_zero=("never_left_zero", "sum"), moved=("moved", "sum"))
    g["frac_never_moved"] = g["never_left_zero"] / g["n"]
    g["frac_moved"] = g["moved"] / g["n"]
    by_layer_frames[label] = g

    mask_rows.append({
        "run": RUN_DISPLAY_NAME[label], "D": int(s.shape[1]),
        "cold-start frozen": int(cold.sum()),
        "never left zero": int(never_left_zero.sum()),
        "moved": int(moved.sum()),
        "frac never moved": float(never_left_zero.float().mean()),
    })

    print(f"[{RUN_DISPLAY_NAME[label]}] -- by layer (conv/fc weights only shown; BN rows all frac_never_moved=0)")
    show = g[g["kind"].isin(["conv", "fc"])]
    display(show.style.format({"frac_never_moved": "{:.3f}", "frac_moved": "{:.3f}"})
            .background_gradient(subset=["frac_never_moved"], cmap="Reds", vmin=0, vmax=1))

display(pd.DataFrame(mask_rows).set_index("run").style.format({"frac never moved": "{:.4f}"}))

for label, g in by_layer_frames.items():
    gg = g[g["kind"].isin(["conv", "fc"])]
    fig, ax = plt.subplots(figsize=(13, 3.8))
    x = np.arange(len(gg))
    ax.bar(x - 0.19, gg["frac_never_moved"], 0.38, label="never left zero", color="#c44e52")
    ax.bar(x + 0.19, gg["frac_moved"], 0.38, label="moved", color="#55a868")
    ax.set_xticks(x); ax.set_xticklabels(gg.index, rotation=75, ha="right", fontsize=7)
    ax.set(ylabel="fraction of layer", title=f"{RUN_DISPLAY_NAME[label]}: conv/fc weight layers", ylim=(0, 1))
    ax.legend(fontsize=8)
    fig.tight_layout(); plt.show()

## 5. How far are the draws from the MAP reference?

`x_ref` in the checkpoint is the cold-start / pruned MAP point. To compare
layers with very different weight scales we express displacement in
**prior-std units**, using the prior precision rebuilt directly from a fresh
`ResNet20` via `build_fan_in_prior_precision_resnet` (same
`prior_std_weight=2.0`, `prior_std_bias=2.0`, `prior_std_bn_weight=1.0`,
`fan_in_scaling=True` the run used). This is the prior term only -- it does
not add the Fisher diagonal, so "prior std" here is a scale normaliser, not
the full posterior-precision scale.

In [ ]:
from sazz.gpu_friendly.models.priors import build_fan_in_prior_precision_resnet

prior_prec = build_fan_in_prior_precision_resnet(
    _ref_module, prior_std_weight=2.0, prior_std_bias=2.0, prior_std_bn_weight=1.0,
    fan_in_scaling=True, dtype=torch.float32, device="cpu",
)
prior_std = prior_prec.clamp(min=1e-12).rsqrt()

per_layer_disp = []
disp_by_coord = {}
for label, r in runs.items():
    ck = r["ckpt"]
    x_ref = ck["x_ref"].cpu()
    s = ck["samples"].cpu()
    abs_disp = (s - x_ref).abs()                 # [n_draws, D]
    scaled = abs_disp / prior_std
    mean_disp = scaled.mean(0)                   # [D]
    disp_by_coord[label] = mean_disp

    ldf = layer_idx.copy()
    ldf["mean_disp"] = mean_disp.numpy()
    bl = ldf.groupby(["kind"], sort=False)["mean_disp"].agg(["mean", "median", "max"])
    bl.columns = pd.MultiIndex.from_product([[f"{RUN_DISPLAY_NAME[label]} (prior std)"], bl.columns])
    per_layer_disp.append(bl)

display(pd.concat(per_layer_disp, axis=1).style.format("{:.4f}").background_gradient(cmap="viridis", axis=None))

# per-layer (full named layer) mean displacement, conv/fc only
for label, r in runs.items():
    ldf = layer_idx.copy()
    ldf["mean_disp"] = disp_by_coord[label].numpy()
    g = ldf.groupby("layer", sort=False).agg(kind=("kind", "first"), mean_disp=("mean_disp", "mean"))
    g = g[g["kind"].isin(["conv", "fc"])]
    fig, ax = plt.subplots(figsize=(13, 3.4))
    ax.bar(np.arange(len(g)), g["mean_disp"], color="#4c72b0")
    ax.set_xticks(np.arange(len(g))); ax.set_xticklabels(g.index, rotation=75, ha="right", fontsize=7)
    ax.set(ylabel="mean |sample - x_ref| (prior std)", title=f"{RUN_DISPLAY_NAME[label]}: per-layer displacement from MAP")
    fig.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(9, 3.6))
for label, r in runs.items():
    ax.hist(disp_by_coord[label].numpy(), bins=120, histtype="step", lw=1.3, label=RUN_DISPLAY_NAME[label])
ax.set(xlabel="per-coordinate mean |sample - x_ref|  (prior std)", ylabel="count",
       title="Displacement from MAP, per coordinate", yscale="log")
ax.legend(fontsize=8)
fig.tight_layout(); plt.show()

## 6. Predictive accuracy and uncertainty

Rebuild the target (CIFAR-10 subset + ResNet-20, matching
`fast_cifar_resnet.py`), push a subsample of draws plus `x_ref` through the
network, and compare posterior-averaged predictions against the single MAP
point.

Load-bearing: the module is put in `.eval()` **before** any forward call, so
BatchNorm uses the frozen running stats (same invariant the sampler ran
under). This is the most expensive cell -- real ResNet-20 forward passes.
Tune `N_UNCERTAINTY_DRAWS` / `N_TEST_EVAL` down if it is slow on CPU.

In [ ]:
from sazz.gpu_friendly.models.model import BayesianModule
from sazz.gpu_friendly.models.priors import build_fan_in_prior_precision_resnet as _bprec
from sazz.gpu_friendly.scripts.fast_cifar_resnet import load_cifar10_subset, BASE_SEED
from sazz.utils.metrics import classification_metrics

N_UNCERTAINTY_DRAWS = 200
N_TEST_EVAL = 1000
DEV = "cuda" if torch.cuda.is_available() else "cpu"
EVAL_DTYPE = torch.float32

print(f"Loading CIFAR-10 test subset (seed={BASE_SEED}, same split as the run) on {DEV} ...")
data = load_cifar10_subset(50_000, N_TEST_EVAL, BASE_SEED, Path("datasets"),
                           dtype=EVAL_DTYPE, device=DEV)
X_test, y_test = data["X_test"], data["y_test"]

perf_rows = []
predictive_probs = {}
for label, r in runs.items():
    ck = r["ckpt"]
    module = ResNet20(activation=ck["activation"]).to(dtype=EVAL_DTYPE, device=DEV)
    module.load_state_dict(_msd, strict=False)   # restore frozen BN running stats
    module.eval()
    names = [n for n, _ in module.named_parameters()]
    shapes = [p.shape for _, p in module.named_parameters()]
    prec = _bprec(module, 2.0, 2.0, 1.0, True, dtype=EVAL_DTYPE, device=DEV)
    bm = BayesianModule.build(module, likelihood="categorical",
                              X=X_test[:2], y=y_test[:2], prior_precision=prec,
                              dtype=EVAL_DTYPE, device=DEV)

    x_ref = ck["x_ref"].to(dtype=EVAL_DTYPE, device=DEV)
    samples = ck["samples"].to(dtype=EVAL_DTYPE)
    idx = torch.randperm(samples.shape[0])[:min(N_UNCERTAINTY_DRAWS, samples.shape[0])]
    sub = samples[idx]

    @torch.no_grad()
    def _probs(beta):
        out = []
        for i in range(0, X_test.shape[0], 256):
            xb = X_test[i:i + 256]
            logits = torch.func.functional_call(bm.module, bm.param_dict_fn(beta.to(DEV)), (xb,))
            out.append(torch.softmax(logits, dim=-1).cpu())
        return torch.cat(out)

    map_probs = _probs(x_ref)
    draw_probs = torch.stack([_probs(b) for b in sub])   # [n_draws, N_test, 10]
    mean_probs = draw_probs.mean(0)

    yt = y_test.cpu()
    predictive_probs[label] = {"map": map_probs, "posterior_draws": draw_probs, "posterior_mean": mean_probs}
    map_metrics = classification_metrics(yt, map_probs)
    post_metrics = classification_metrics(yt, mean_probs)
    p_true_per_draw = draw_probs.gather(-1, yt.long().view(1, -1, 1).expand(draw_probs.shape[0], -1, 1)).squeeze(-1)
    disagreement = p_true_per_draw.std(dim=0).mean().item()

    perf_rows.append({
        "run": RUN_DISPLAY_NAME[label],
        "map accuracy": map_metrics["accuracy"], "map log_lik": map_metrics["log_lik"],
        "map entropy": map_metrics["entropy"],
        "posterior accuracy": post_metrics["accuracy"], "posterior log_lik": post_metrics["log_lik"],
        "posterior entropy": post_metrics["entropy"],
        "ckpt test_accuracy": ck.get("test_accuracy"),
        "mean draw-to-draw P(true class) std": disagreement,
    })

perf_df = pd.DataFrame(perf_rows).set_index("run")
display(perf_df.style.format("{:.4f}"))

print(
    "Read: if 'posterior entropy' > 'map entropy' at matched accuracy, the sampler is\n"
    "adding epistemic-uncertainty signal. 'mean draw-to-draw P(true class) std' near 0\n"
    "means the draws behave like near-duplicates of x_ref on the test set."
)

In [ ]:
# Reliability curve + entropy split for correct vs wrong predictions.
yt = y_test.cpu().long()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
for label in runs:
    mean_probs = predictive_probs[label]["posterior_mean"]
    conf, pred = mean_probs.max(-1)
    correct = (pred == yt)
    bins = torch.linspace(0, 1, 11)
    xs, accs = [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf >= lo) & (conf < hi)
        if m.any():
            xs.append(((lo + hi) / 2).item())
            accs.append(correct[m].float().mean().item())
    axes[0].plot(xs, accs, marker="o", label=RUN_DISPLAY_NAME[label])

    ent = -(mean_probs.clamp_min(1e-12) * mean_probs.clamp_min(1e-12).log()).sum(-1)
    axes[1].hist(ent[correct].numpy(), bins=40, histtype="step", lw=1.4,
                 label=f"{RUN_DISPLAY_NAME[label]} correct")
    axes[1].hist(ent[~correct].numpy(), bins=40, histtype="step", lw=1.4, ls="--",
                 label=f"{RUN_DISPLAY_NAME[label]} wrong")

axes[0].plot([0, 1], [0, 1], "k:", lw=1)
axes[0].set(xlabel="posterior-mean confidence", ylabel="empirical accuracy",
            title="Reliability (posterior-averaged)")
axes[0].legend(fontsize=8)
axes[1].set(xlabel="predictive entropy", ylabel="count", title="Entropy: correct vs wrong", yscale="log")
axes[1].legend(fontsize=7)
fig.tight_layout(); plt.show()

## 7. Takeaways

Fill in after running -- the questions this notebook was built to answer:

1. **File** (sec 1): one Zig-Zag checkpoint, `samples [n_draws, 272474]`,
   `x_ref`, `cold_start_mask`, scalar summaries, `grid_t_max_log`.
   `diagnostics` is `None`; no chunk dir -- no per-iteration event mix.
2. **Layer map** (sec 2): sparsity is a conv/fc-weight story -- BN gamma/beta
   and the fc bias are never freezable.
3. **Stem conv** (sec 3): does the sampler zero whole (filter, RGB-channel)
   slabs? Where does posterior activation std land on a real image -- on
   object structure, or flat?
4. **Sparsity** (sec 4): did the path thaw or freeze relative to the
   cold-start prune? Which conv layers never move?
5. **Displacement** (sec 5): are the draws exploring around `x_ref`, and
   which layers move most in prior-std units (early conv vs deep conv vs fc)?
6. **Predictions** (sec 6): is the posterior-averaged prediction more
   usefully uncertain than the MAP point at the same accuracy? Is it better
   calibrated, and is entropy higher on the wrong predictions?